Scenario: Automating Invoice Processing with Azure Form Recognizer

Context

You’re a data engineer at an e‑commerce company. Your finance team receives

hundreds of invoices in PNG format from vendors. Manually typing invoice
details into the ERP system is slow and error‑prone. You decide to use

Azure

Form Recognizer’s prebuilt invoice model to automate extraction.

In [1]:
import requests
import time
from google.colab import files

#Step 1: Upload file
print("Upload your invoice file...")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
data = uploaded[file_name]

# Step 2: Azure Config
endpoint = "https://sanskriti-docu.cognitiveservices.azure.com"   #endpoint
key = "A3voG5o********************"  #enter your correct key

analyze_url = endpoint + "/formrecognizer/documentModels/prebuilt-invoice:analyze?api-version=2023-07-31"

headers = {
    "Ocp-Apim-Subscription-Key": key,
    "Content-Type": "image/jpeg"   # <-- because your file is jpeg
}

# Step 3: Send request
print("Analyzing invoice...")
response = requests.post(analyze_url, headers=headers, data=data)

if response.status_code != 202:
    print("Error:", response.text)
    exit()

operation_url = response.headers["operation-location"]
print("Processing started...")

# Step 4: Polling (VERY IMPORTANT)
while True:
    result_response = requests.get(operation_url, headers={
        "Ocp-Apim-Subscription-Key": key
    })

    result_json = result_response.json()
    status = result_json["status"]

    print(f"Status: {status}")

    if status == "succeeded":
        print("Analysis completed!\n")
        break
    elif status == "failed":
        print("Analysis failed")
        exit()

    time.sleep(2)

# Step 5: Extract useful fields
documents = result_json["analyzeResult"]["documents"]

if documents:
    fields = documents[0]["fields"]

    print("Extracted Invoice Data:\n")

    for key, value in fields.items():
        if "content" in value:
            print(f"{key}: {value['content']}")
else:
    print("No data found.")

# try:
#     print(response.json())
# except requests.exceptions.JSONDecodeError as e:
#     print(f"Error decoding JSON: {e}")
#     print(f"Raw response content: {response.text}")

Upload your invoice file...


Saving Screenshot 2026-04-10 at 8.45.15 PM.png to Screenshot 2026-04-10 at 8.45.15 PM.png
Analyzing invoice...
Processing started...
Status: running
Status: succeeded
Analysis completed!

Extracted Invoice Data:

AmountDue: $
20.27
InvoiceDate: 8/8/24
InvoiceId: 76543
InvoiceTotal: $
20.27
SubTotal: $
18.00
TotalTax: $
1.28
VendorName: LOGO
